# RELIC — Entrenamiento Curricular a Gran Escala (Fase A/B)
## ESTADO: PENDIENTE DE EJECUTAR EN GOOGLE COLAB / KAGGLE (NO EJECUTAR LOCALMENTE)

> **Regla Operativa:** Este notebook debe ejecutarse en acelerador GPU externo (T4/V100/A100). No ejecutar localmente en máquina de recursos limitados.

### Diagnósticos Obligatorios Incluidos:
1. **Curvas de pérdida:** Cross-Entropy con gradiente enmascarado solo en tokens objetivo tras `<asistente>`.
2. **% de JSON válido:** Tasa de decodificaciones sintácticamente parseables por `json.loads` en cada época.
3. **Acierto DENTRO de plantillas de entrenamiento (In-Distribution):** Precisión en plantillas vistas.
4. **Acierto en plantillas de Holdout (Out-of-Distribution):** Generalización por construcción (split de clústeres).
5. **Evaluación multi-semilla:** 5 semillas `[42, 1337, 2026, 7777, 9999]` con reporte de media y desvío estándar.

In [ ]:
# 1. Entorno y GPU
!nvidia-smi
!pip install --quiet torch safetensors tqdm matplotlib numpy

In [ ]:
import os, json, math, random, re
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from safetensors.torch import save_file
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Acelerador: {device}")

In [ ]:
# 2. Arquitectura RELIC-10M (Conforme a relic/config.py y relic_core::model)
from dataclasses import dataclass

@dataclass
class RelicConfig:
    vocab_size: int = 8192
    hidden_size: int = 288
    intermediate_size: int = 768
    num_hidden_layers: int = 6
    num_attention_heads: int = 6
    num_key_value_heads: int = 2
    head_dim: int = 48
    max_position_embeddings: int = 512
    rope_theta: float = 10000.0
    rms_norm_eps: float = 1e-6

config = RelicConfig()
print(f"Configuración RELIC cargada: {config.num_hidden_layers} capas, {config.hidden_size} dim, vocab {config.vocab_size}")

In [ ]:
# 3. Módulos Neuronales (RMSNorm, SwiGLU, GQA, Bloques Transformer)
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        var = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(var + self.eps) * self.weight

class SwiGLUMLP(nn.Module):
    def __init__(self, h_size: int, inter_size: int):
        super().__init__()
        self.gate_proj = nn.Linear(h_size, inter_size, bias=False)
        self.up_proj = nn.Linear(h_size, inter_size, bias=False)
        self.down_proj = nn.Linear(inter_size, h_size, bias=False)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(nn.functional.silu(self.gate_proj(x)) * self.up_proj(x))

class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg: RelicConfig):
        super().__init__()
        self.num_heads = cfg.num_attention_heads
        self.num_kv = cfg.num_key_value_heads
        self.head_dim = cfg.head_dim
        self.q_proj = nn.Linear(cfg.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.hidden_size, self.num_kv * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.hidden_size, self.num_kv * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, cfg.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=cfg.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=cfg.rms_norm_eps)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, _ = x.shape
        q = self.q_norm(self.q_proj(x).view(B, S, self.num_heads, self.head_dim))
        k = self.k_norm(self.k_proj(x).view(B, S, self.num_kv, self.head_dim))
        v = self.v_proj(x).view(B, S, self.num_kv, self.head_dim)
        k = k.repeat_interleave(self.num_heads // self.num_kv, dim=2)
        v = v.repeat_interleave(self.num_heads // self.num_kv, dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        attn = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(attn.transpose(1, 2).contiguous().view(B, S, -1))

class RelicBlock(nn.Module):
    def __init__(self, cfg: RelicConfig):
        super().__init__()
        self.input_layernorm = RMSNorm(cfg.hidden_size, eps=cfg.rms_norm_eps)
        self.self_attn = GroupedQueryAttention(cfg)
        self.post_attention_layernorm = RMSNorm(cfg.hidden_size, eps=cfg.rms_norm_eps)
        self.mlp = SwiGLUMLP(cfg.hidden_size, cfg.intermediate_size)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = x + self.self_attn(self.input_layernorm(x))
        return h + self.mlp(self.post_attention_layernorm(h))

class RelicForCausalLM(nn.Module):
    def __init__(self, cfg: RelicConfig):
        super().__init__()
        self.config = cfg
        self.embed_tokens = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.layers = nn.ModuleList([RelicBlock(cfg) for _ in range(cfg.num_hidden_layers)])
        self.norm = RMSNorm(cfg.hidden_size, eps=cfg.rms_norm_eps)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        x = self.embed_tokens(ids)
        for layer in self.layers: x = layer(x)
        return self.lm_head(self.norm(x))

m = RelicForCausalLM(config)
print(f"Parámetros inicializados: {sum(p.numel() for p in m.parameters()):,}")

In [ ]:
# 4. Carga y Tokenización con ByteTokenizer
class RelicByteTokenizer:
    PAD_ID, BOS_ID, EOS_ID, USER_ID, ASSISTANT_ID = 0, 1, 2, 3, 4
    BYTE_OFFSET = 5
    def encode(self, text: str) -> list[int]:
        return [b + self.BYTE_OFFSET for b in text.encode('utf-8')]
    def decode(self, ids: list[int]) -> str:
        bs = bytes([i - self.BYTE_OFFSET for i in ids if self.BYTE_OFFSET <= i <= (255 + self.BYTE_OFFSET)])
        return bs.decode('utf-8', errors='replace')

tok = RelicByteTokenizer()

class CurriculumDataset(Dataset):
    def __init__(self, jsonl_path: str, split_key: str, max_len: int = 256):
        self.samples = []
        self.max_len = max_len
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                d = json.loads(line)
                if split_key.lower() in d.get('split', '').lower():
                    self.samples.append(d)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        item = self.samples[idx]
        user_ids = [tok.USER_ID] + tok.encode(item['prompt']) + [tok.ASSISTANT_ID]
        rir_str = json.dumps(item['gold_rir'], ensure_ascii=False, separators=(',', ':'))
        target_ids = tok.encode(rir_str) + [tok.EOS_ID]
        seq = (user_ids + target_ids)[:self.max_len]
        labels = ([-100] * len(user_ids) + target_ids)[:self.max_len]
        pad_len = self.max_len - len(seq)
        seq += [tok.PAD_ID] * pad_len
        labels += [-100] * pad_len
        return {'input_ids': torch.tensor(seq, dtype=torch.long), 'labels': torch.tensor(labels, dtype=torch.long), 'gold_rir': item['gold_rir'], 'prompt': item['prompt']}

ds_file = 'curriculum_dataset_v2_20k.jsonl'
print(f"Cargando dataset desde {ds_file}...")
train_ds = CurriculumDataset(ds_file, 'train')
holdout_ds = CurriculumDataset(ds_file, 'holdout')
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
print(f"Dataset cargado: {len(train_ds)} train, {len(holdout_ds)} holdout.")

In [ ]:
# 5. Diagnósticos de Decodificación: % JSON Válido y Exact Match
def evaluate_diagnostics(model, sample_batch, max_tokens=120):
    model.eval()
    valid_json_count = 0
    exact_match_count = 0
    total = min(len(sample_batch), 50)
    with torch.no_grad():
        for i in range(total):
            item = sample_batch[i]
            user_ids = [tok.USER_ID] + tok.encode(item['prompt']) + [tok.ASSISTANT_ID]
            curr = torch.tensor([user_ids], dtype=torch.long, device=device)
            gen_ids = []
            for _ in range(max_tokens):
                nxt = torch.argmax(model(curr)[:, -1, :], dim=-1).item()
                if nxt == tok.EOS_ID: break
                gen_ids.append(nxt)
                curr = torch.cat([curr, torch.tensor([[nxt]], device=device)], dim=1)
            text_out = tok.decode(gen_ids)
            try:
                p_json = json.loads(text_out)
                valid_json_count += 1
                if p_json == item['gold_rir']:
                    exact_match_count += 1
            except Exception:
                pass
    return (valid_json_count / total) * 100.0, (exact_match_count / total) * 100.0

print("Función de diagnósticos lista.")

In [ ]:
# 6. Bucle de Entrenamiento Multi-Semilla con Diagnósticos por Época
SEEDS = [42, 1337, 2026, 7777, 9999]
multi_seed_results = {}

for seed in SEEDS:
    print(f"\n=== EJECUTANDO SEMILLA {seed} ===")
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    
    model = RelicForCausalLM(config).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.01)
    crit = nn.CrossEntropyLoss(ignore_index=-100)
    
    history = {'train_loss': [], 'json_valid_train': [], 'em_train': [], 'em_holdout': []}
    
    for ep in range(1, 11):
        model.train()
        tot_loss = 0.0
        for b in train_loader:
            ids = b['input_ids'].to(device)
            lbls = b['labels'].to(device)
            opt.zero_grad()
            logits = model(ids)
            loss = crit(logits[..., :-1, :].reshape(-1, config.vocab_size), lbls[..., 1:].reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot_loss += loss.item()
        avg_loss = tot_loss / len(train_loader)
        history['train_loss'].append(avg_loss)
        
        # Diagnósticos en muestra de Train e Holdout
        pct_json, em_tr = evaluate_diagnostics(model, train_ds.samples)
        _, em_ho = evaluate_diagnostics(model, holdout_ds.samples)
        history['json_valid_train'].append(pct_json)
        history['em_train'].append(em_tr)
        history['em_holdout'].append(em_ho)
        
        print(f"Época {ep:02d}/10 | Loss: {avg_loss:.4f} | JSON Válido: {pct_json:.1f}% | EM Train: {em_tr:.1f}% | EM Holdout: {em_ho:.1f}%")
    
    multi_seed_results[seed] = history

print("\n[OK] Entrenamiento multi-semilla completado.")

In [ ]:
# 7. Gráficas Consolidadas de Diagnóstico
plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1)
for s in SEEDS: plt.plot(multi_seed_results[s]['train_loss'], label=f'Seed {s}')
plt.title('Curva de Pérdida Train'); plt.xlabel('Época'); plt.ylabel('Loss'); plt.legend()

plt.subplot(1, 3, 2)
for s in SEEDS: plt.plot(multi_seed_results[s]['json_valid_train'], label=f'Seed {s}')
plt.title('% JSON Válido'); plt.xlabel('Época'); plt.ylabel('% Válido')

plt.subplot(1, 3, 3)
for s in SEEDS:
    plt.plot(multi_seed_results[s]['em_train'], label=f'Tr {s}')
    plt.plot(multi_seed_results[s]['em_holdout'], '--', label=f'Ho {s}')
plt.title('Acierto Exacto (Solid=Train, Dash=Holdout)'); plt.xlabel('Época'); plt.ylabel('EM %')
plt.tight_layout(); plt.show()